# Phase 5a: Modeling Dataset and Evaluation Design

This notebook prepares the modeling dataset for predictive analysis of `unmet_need_pc` using the enriched Eurostat panel. No model is estimated here.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'panel_features_v2-3.csv'
df = pd.read_csv(DATA_PATH)
aggregate_geo_codes = {'EU27_2020', 'EU28', 'EU15', 'EA', 'EA19', 'EA20'}
df = df.loc[~df['geo'].isin(aggregate_geo_codes)].copy()
df.head()

,geo,year,unmet_need_pc,status,gdp_per_capita_eur,unemployment_rate_pc,poverty_or_social_exclusion_pc,government_health_expenditure_gdp_pc,compulsory_health_financing_gdp_pc,physicians_per_100k,hospital_beds_per_100k,oop_health_expenditure_share_pc,gini_income,long_term_unemployment_rate_pc,log_gdp_per_capita,log_oop_health_expenditure_share,gdp_per_capita_eur_lag1,unemployment_rate_pc_lag1,poverty_or_social_exclusion_pc_lag1,physicians_per_100k_lag1,gdp_per_capita_growth,unemployment_rate_change
0,AL,2017,13.1,NaN,4100.0,NaN,58.5,NaN,NaN,NaN,NaN,NaN,36.8,NaN,8.318742,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AL,2018,14.8,NaN,4540.0,NaN,53.9,NaN,NaN,NaN,NaN,NaN,35.4,NaN,8.420682,NaN,4100.0,NaN,58.5,NaN,0.107317,NaN
2,AL,2019,14.6,NaN,4880.0,NaN,50.7,NaN,NaN,NaN,NaN,NaN,34.3,NaN,8.492900,NaN,4540.0,NaN,53.9,NaN,0.074890,NaN
3,AL,2020,10.6,NaN,4710.0,NaN,46.2,NaN,NaN,NaN,NaN,NaN,33.2,NaN,8.457443,NaN,4880.0,NaN,50.7,NaN,-0.034836,NaN
4,AL,2021,10.7,NaN,5420.0,NaN,46.6,NaN,NaN,NaN,NaN,NaN,33.0,NaN,8.597851,NaN,4710.0,NaN,46.2,NaN,0.150743,NaN


## Load and Inspect Data

The enriched panel contains 608 rows and 22 columns. It covers 37 distinct national geo codes. The year variable has 18 distinct values, ranging from 2008 to 2025.

In [2]:
data_summary = {
    'rows': len(df),
    'columns': df.shape[1],
    'distinct_geo_codes': df['geo'].nunique(),
    'distinct_years': df['year'].nunique(),
    'min_year': int(df['year'].min()),
    'max_year': int(df['year'].max()),
}
data_summary

{'rows': 608,
 'columns': 22,
 'distinct_geo_codes': 37,
 'distinct_years': 18,
 'min_year': 2008,
 'max_year': 2025}

In [3]:
outcome = 'unmet_need_pc'

original_controls = [
    'gdp_per_capita_eur',
    'unemployment_rate_pc',
    'poverty_or_social_exclusion_pc',
    'government_health_expenditure_gdp_pc',
    'compulsory_health_financing_gdp_pc',
]

structural_variables = [
    'physicians_per_100k',
    'hospital_beds_per_100k',
    'oop_health_expenditure_share_pc',
    'gini_income',
    'long_term_unemployment_rate_pc',
]

engineered_variables = [
    'log_gdp_per_capita',
    'gdp_per_capita_growth',
    'unemployment_rate_change',
    'physicians_per_100k_lag1',
]

features = original_controls + structural_variables + engineered_variables
missing_features = [col for col in features if col not in df.columns]
missing_features

[]

## Candidate Feature List

Outcome: `unmet_need_pc`

Original controls:
- `gdp_per_capita_eur`
- `unemployment_rate_pc`
- `poverty_or_social_exclusion_pc`
- `government_health_expenditure_gdp_pc`
- `compulsory_health_financing_gdp_pc`

Structural variables:
- `physicians_per_100k`
- `hospital_beds_per_100k`
- `oop_health_expenditure_share_pc`
- `gini_income`
- `long_term_unemployment_rate_pc`

Engineered variables:
- `log_gdp_per_capita`
- `gdp_per_capita_growth`
- `unemployment_rate_change`
- `physicians_per_100k_lag1`

The final feature set contains 14 variables. `physicians_per_100k_lag1` is used as the available health-system lag variable.

In [4]:
feature_categories = {}
for name in original_controls:
    feature_categories[name] = 'Original control'
for name in structural_variables:
    feature_categories[name] = 'Structural variable'
for name in engineered_variables:
    feature_categories[name] = 'Engineered variable'

feature_set_md = [
    '# Phase 5a model feature set',
    '',
    f'Outcome: `{outcome}`',
    '',
    'The selected feature set uses the original controls, the five verified structural variables, and four interpretable engineered variables present in the enriched panel.',
    '',
    '| Category | Feature |',
    '|---|---|',
]
for feature in features:
    feature_set_md.append(f'| {feature_categories[feature]} | `{feature}` |')
feature_set_md.extend(['', 'Feature names only:', ''])
feature_set_md.extend([f'- `{feature}`' for feature in features])

with open(PROJECT_ROOT / 'outputs' / 'model_feature_set_5a.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(feature_set_md) + '\n')

In [5]:
df_model = df.dropna(subset=[outcome] + features).copy()
df_model = df_model.sort_values(['year', 'geo']).reset_index(drop=True)
df_model.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'modeling_dataset_5a.csv', index=False)

model_summary = {
    'rows': len(df_model),
    'distinct_countries': df_model['geo'].nunique(),
    'distinct_years': df_model['year'].nunique(),
    'min_year': int(df_model['year'].min()),
    'max_year': int(df_model['year'].max()),
}
model_summary

{'rows': 230,
 'distinct_countries': 27,
 'distinct_years': 9,
 'min_year': 2015,
 'max_year': 2023}

## Complete-Case Modeling Dataset

The complete-case modeling dataset contains 230 rows. It covers 27 distinct countries and 9 distinct years, ranging from 2015 to 2023.

In [6]:
df_model.groupby('year').size().rename('rows').reset_index()

,year,rows
0,2015,25
1,2016,25
2,2017,26
3,2018,26
4,2019,26
5,2020,26
6,2021,26
7,2022,26
8,2023,24


In [7]:
train_years = list(range(2015, 2019))
valid_years = list(range(2019, 2021))
test_years = list(range(2021, 2024))

def assign_split(year):
    if year in train_years:
        return 'train'
    if year in valid_years:
        return 'valid'
    if year in test_years:
        return 'test'
    return np.nan

df_model['split'] = df_model['year'].astype(int).map(assign_split)
assert df_model['split'].notna().all()

df_model.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'modeling_dataset_5a_with_splits.csv', index=False)
df_model['split'].value_counts().reindex(['train', 'valid', 'test'])

split
train    102
valid     52
test      76
Name: count, dtype: int64

## Evaluation Scheme

Train years: 2015-2018 (102 rows).  
Validation years: 2019-2020 (52 rows).  
Test years: 2021-2023 (76 rows).

This split uses the earliest complete-case observations for training, reserves the next two years for model selection, and keeps the three most recent complete-case years for final testing. This is reasonable for a time-aware panel because later observations are not used to tune models evaluated on earlier years.

## Evaluation Metrics

Mean Absolute Error is the average absolute difference between observed and predicted `unmet_need_pc`.

Root Mean Squared Error is the square root of the average squared difference between observed and predicted `unmet_need_pc`.

R-squared is the proportion of variance in `unmet_need_pc` explained by predictions compared with a constant mean model.